# 🦜🔗 LangChain Fundamentals: A Hands-On Tour
Welcome!  
This notebook gives you a fast-paced, practical tour of the **core components that make LangChain powerful**.  
You’ll learn how to talk to AI, make it use tools, and even make it read your own data.

## 📚 What We Will Cover

### **1. Prompt Templates**
Learn how to communicate with the model efficiently using reusable and structured prompts.
### **2. Chat Models**
These are the *brains* of your application—LLMs that generate responses, reason, and follow instructions.
### **3. LCEL (Chaining)**
Connect multiple steps together to build workflows.  
### **4. Structured Output**
Receive clean, typed outputs (like JSON or Pydantic models) for smooth integration with code.
### **5. Tool Calling**
Giving the AI a calculator, API access, or other utilities. 


Set your OpenAI key (skip if already in your shell env).

In [1]:
# Load settings needed by later cells. Run the notebook from top to bottom.

# Import a helper that reads settings from a .env file.
from dotenv import load_dotenv
# Load values such as your API key into this Python session.
# The path is relative to the current working folder; this does not print the key.
load_dotenv(dotenv_path=".env")

True

In [1]:
# Optional example, currently disabled: lines starting with # do not run.
# os provides access to environment settings. Avoid printing API keys,
# because notebook outputs may be saved or shared.

#import os
#print(os.getenv("OPENAI_API_KEY"))

## 1) Prompt Templates
Imagine ordering at a subway shop. The sentence structure is always the same: 
*"I would like a **{Size}** sandwich on **{Bread_Type}**."*
You don't invent a new way to speak every time you order lunch; you just change the specific ingredients. Prompt Templates are exactly that—reusable blueprints where we dynamic variables (like "6-inch" or "Italian Herb") without rewriting the whole request.

###  Code Context
In the code below:
* `ChatPromptTemplate`: This is our blueprint.
* `{user_query}` and `{todays_date}`: These are the **variables** (the blanks we will fill in later).
* We use a System Prompt to tell the AI *who* it is (a Date Assistant) before it answers.


![Diagram](./images/01_img.png)

In [3]:
# Build a reusable set of instructions and a question for the AI.

# Read the system instructions from the companion Python file.
from langchain_prompts import DATE_ASSISTANT_SYSTEM_PROMPT
# Import Python's date and time helper.
from datetime import datetime
# Import a helper for building chat messages with fill-in-the-blank fields.
from langchain_core.prompts import ChatPromptTemplate

# A prompt template is a reusable message outline.
# The system message sets the instructions; the human message contains the question.
# Names inside braces, such as {user_query}, are placeholders filled in later.
prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_ASSISTANT_SYSTEM_PROMPT),
    ("human", "{user_query}")
])

# Store a sample question. Triple quotes allow text to span several lines.
example_user_query = """
What is tomorrow's date?
"""

# Fill the placeholders to create the actual messages. This does not call the AI.
# datetime.now() uses the computer's local date and time.
# strftime formats that date as year-month-day, for example 2026-10-15.
messages = prompt.format_messages(user_query = example_user_query, 
                                    todays_date = datetime.now().strftime("%Y-%m-%d"))  # Shows message objects ready for a chat model

# As the last expression in this cell, this displays the message objects in Jupyter.
messages


# Other message types represent an AI reply or a result returned by a tool.
# Other Message Objects
    # System messages
    # Human messages
    # AIMessage
    # ToolMessage
    

[SystemMessage(content="\n\nYou are a helpful and accurate date assistant. \nYou are given a user's request, and today's date. Your job is to accurately infer the date in YYYY-MM-DD format from the user's request.\n\ntoday's date : 2026-10-16\n\nchat_history:\n\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="\nWhat is tomorrow's date?\n", additional_kwargs={}, response_metadata={})]

In [4]:
# Inspect the prepared text before sending it to the model.
# Python counts positions from zero: [0] is the first message, [1] the second.
# .content gives the text inside each message object.

print(messages[0].content)
print(messages[1].content)



You are a helpful and accurate date assistant. 
You are given a user's request, and today's date. Your job is to accurately infer the date in YYYY-MM-DD format from the user's request.

today's date : 2026-10-16

chat_history:



What is tomorrow's date?



## 2) Chat Models (OpenAI)

Think of Chat Models as different brands of TVs (Sony, Samsung, LG). Each one works differently on the inside, but LangChain gives you a **"Universal Remote"** to control them all.
This "Universal Remote" is called the **Runnable Interface**. It means you can press the "Play" button (`invoke`) on *any* model, and it will work exactly the same way. You don't need to learn a new remote control just because you bought a new TV.

### 🧠 Interface Concepts
all Chat Models implement the **Runnable Interface**. This guarantees they all speak the same language:
1.  **Standard Inputs:** They all accept a list of `Messages` (System, Human, AI).
2.  **Standard Outputs:** They all return an `AIMessage`.
3.  **Standard Methods:**
    * `.invoke()`: Send a message and wait for the full answer.
    * `.stream()`: Get the answer word-by-word (like a typewriter).
    * `.batch()`: Send 50 questions at once and get 50 answers back.

###  Code Context
In the code below:
* `init_chat_model`: A universal function to load a model.
* `model="gpt-4o-mini"`: We are selecting a specific, fast model from OpenAI.
* `temperature=0`: We set this to 0 to make the AI **factual and consistent**. Higher numbers (up to 1) make it more creative and random.

- Supports over 100 model providers : https://docs.langchain.com/oss/python/integrations/chat
- https://docs.langchain.com/oss/python/langchain/models

![Diagram](./images/02_img.png)

In [5]:
# Create the model connection used in later cells.
# The model name selects which AI to use. temperature=0 requests less varied
# answers, but does not guarantee identical or correct answers.
# Actual requests need credentials and network access and may incur API charges.

from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", temperature=0) # Tip: Try other models/model providers

## 3) LLM Invocation  

* **Invoke:** This is like sending a text message and waiting for the reply. You don't see anything until the whole message arrives.
* **Stream:** This is like watching someone type in real-time. You see the words appear one by one. This feels much faster to a user.

###  Code Context
In the code below:
* `llm.invoke(messages)`: Sends our formatted prompt to the AI and waits for the full `AIMessage` response.
* `llm.stream(messages)`: Returns a generator that prints tokens as they arrive.

📚 **[Docs](https://docs.langchain.com/oss/python/langchain/models#invocation)**

![Diagram](./images/03_img.png)

In [6]:
# Send the prepared messages to the AI service and wait for the complete reply.
# The reply is an object containing text and other information.

# Invoke : 
response = llm.invoke(messages)
# Display the whole reply object so you can inspect its details.
response


AIMessage(content="Tomorrow's date is 2026-10-17.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 73, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8794d3bcd1', 'id': 'chatcmpl-EZKa2IqjK4Q29ZL1GOfXSqy3FvCyd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a140d8-911b-7e73-b3d8-dfe1b7560ed5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 12, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
# Show just the text of the AI reply, without the surrounding metadata.

print(response.content)

Tomorrow's date is 2026-10-17.


## 3.1) Streaming (Real-time Output)

Imagine you hired a ghostwriter to write a story.
* **Invoke (Standard):** The writer goes into a private room, locks the door, writes the entire story, and comes out 20 minutes later to hand you the finished manuscript. You sit in silence waiting the whole time.
* **Stream (Real-time):** The writer sits right in front of you and types. You see every letter appear on the screen the moment they press the key. The story takes the same amount of time to finish, but it **feels instant** because you are engaged immediately.

### Technical Concept
LLMs don't "think" of a whole sentence at once; they generate text **one token (word part) at a time**.
* **The Bottleneck:** By default (`.invoke()`), the program buffers (holds) all these tokens in memory until the model is completely finished. This creates **Latency** (the delay between asking and seeing the answer).
* **The Solution:** The `.stream()` method bypasses this buffer. It creates a Python **Generator** that yields each token (`AIMessageChunk`) the exact millisecond it is created. This drastically lowers the "Time to First Token."

### Code Context
In the code below:
* `llm.stream(messages)`: This replaces `.invoke()`. It returns an iterable stream, not a final string.
* `print(..., end="", flush=True)`: We force Python to print *immediately* without moving to a new line, creating that smooth "typewriter" effect.

📚 **[Docs: Streaming](https://docs.langchain.com/oss/python/langchain/streaming)**


In [8]:
# Streaming shows the reply as pieces arrive instead of waiting for the full answer.

# Streaming : 

# The loop handles each incoming piece in turn.
for chunk in llm.stream(messages):
    # end="" keeps pieces on the same line; flush=True displays them immediately.
    print(chunk.content, end="", flush=True)


Tomorrow's date is 2026-10-17.

## 4) Chaining with LCEL (LangChain Expression Language)

Imagine a factory assembly line:
* **Station A:** Puts dough on the belt (The Prompt).
* **Station B:** Bakes the dough (The Model).
* **Station C:** Packages the bread (The Output Parser).

LCEL lets us pipe (`|`) these steps together. The output of one step automatically becomes the input of the next.

###  Code Context
In the code below:
* `chain = prompt | llm | StrOutputParser()`
* The `|` symbol is the magic pipe.
* `StrOutputParser`: Converts the complex AI message object directly into a simple string so we don't have to do `print(response.content)`.

📚 **[Docs: LCEL Conceptual Guide](https://python.langchain.com/docs/concepts/lcel/)** *(the LCEL guide still lives on the legacy docs site — in LangChain 1.x, piping is fully supported for simple prompt → model → parser chains, but agents are the recommended pattern for anything that loops or branches)*

![Diagram](./images/04_img.png)

In [9]:
# Combine preparation, the AI request, and text extraction into one workflow.

# This parser extracts the text from the model's reply object.
from langchain_core.output_parsers import StrOutputParser

# The | symbol connects steps from left to right:
# fill the prompt, ask the model, then extract its text reply.
chain = prompt | llm | StrOutputParser()


# Run the entire workflow using a dictionary: each name is paired with a value.
# This makes a new AI request using the current local date.
print(chain.invoke({ 'user_query' : example_user_query, 
                    'todays_date' : datetime.now().strftime("%Y-%m-%d") }))


Tomorrow's date is 2026-10-17.


## 5) Structured Output

Usually, AI likes to chat and write paragraphs. But sometimes, you want a form filled out, not a conversation.
If you ask for a date range, you don't want *"Sure! The start date is..."*. You want `{"start": "2025-01-01", "end": "2025-01-05"}`.
Structured output forces the AI to stop chatting and generate data matching a specific schema.

###  Code Context
In the code below:
* `class DateRange(BaseModel)`: We define the "Form" we want the AI to fill out using Pydantic.
* `llm.with_structured_output(DateRange)`: We tell the LLM, "Your output **must** match this class."
* The result is an actual Python object, not a string.

📚 **[Docs: Structured Output](https://docs.langchain.com/oss/python/langchain/models#structured-output)**

![Diagram](./images/05_img.png)

In [10]:
# Structured output asks the model for named fields instead of a free-form answer.
# These are alternative ways to describe the expected fields.
# Pydantic can validate data; TypedDict mainly describes dictionary fields for
# development tools. Dataclasses and JSON Schema are other supported formats.

# Supports
    # - pydantic
    # - Typedict
    # - Dataclass 
    # - Json schema

In [11]:
# Extract a start and end date from an everyday request.
# This cell replaces the earlier llm, prompt, and chain variables with new examples.

# Import helpers for describing data types. Some imports support optional examples.
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_prompts import DATE_EXTRACTOR_PROMPT_TEMPLATE

# The commented Pydantic example below is an alternative and does not run.
# # You can define structure with Pydantic
class DateRange(BaseModel):
     start_date: str = Field(default = None, description="Start Date of the date range")
     end_date: str = Field(default = None, description="End Date of the date range")

# OR you can define structure with TypedDict
# Describe a dictionary with two named text fields.
# str means text. These annotations alone do not validate the dates.
#class DateRange(TypedDict):
#    start_date: str
#    end_date: str


# Create the base AI model for the date extraction task.
llm = init_chat_model(model="gpt-4o-mini", temperature=0)

# Configure the model to return the fields described by DateRange.
structured_llm = llm.with_structured_output(DateRange)

# Prepare extraction instructions and a placeholder for the person's request.
prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_EXTRACTOR_PROMPT_TEMPLATE),
    ("human", "{user_query}")
])


# Set the sample request and today's date so relative dates have a reference point.
user_query = "I need time off for the next 4 days"
todays_date = datetime.now().strftime("%Y-%m-%d")


# Connect the prompt to the model configured for structured output.
chain = prompt | structured_llm 

# Send the request and reference date through the workflow.
# The returned dictionary should contain start_date and end_date.
llm_response = chain.invoke({'todays_date': todays_date, 'user_query': user_query})

# Display the extracted fields in the notebook.
llm_response


DateRange(start_date='2026-10-17', end_date='2026-10-20')

## 6) Tool Calling

LLMs are great at writing poetry, but they are often bad at math and they don't know the current weather.
Tool calling is like giving the AI **specialized gadgets** to help it do its job.
You tell the AI: *"I am giving you a calculator tool. If you see a math problem, don't guess—use the tool."*
The AI decides **when** it needs help and **which** gadget to pick to solve the problem.

###  Code Context
In the code below:
* `@tool`: This is a decorator. It puts a "sticker" on your Python function that says *"Hey AI, you are allowed to use this!"*
* `llm.bind_tools(tools)`: This effectively hands the manual of available tools to the "Brain."
* **Important:** When the model uses a tool, it doesn't return a text answer immediately. It returns a `tool_call` (a request to run the code). You (the code) must then run that function and give the answer back to the AI.

📚 **[Docs: Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)**


![Diagram](./images/06_img.png)

In [12]:
# Give the AI information about a calculation it can request.

# Import the decorator that exposes a Python function as an AI tool.
from langchain.tools import tool
# HumanMessage represents a message from the person asking the question.
from langchain.messages import HumanMessage
# Python's math module supplies pi for the circle area formula.
import math

# The decorator marks the following function as a tool.
# Its name, type hints, and description help the model understand how to use it.
@tool
def circle_area(radius: float) -> float: 
    """Return the area of a circle for a given radius."""
    # Calculate area = pi times radius squared. float means a decimal number.
    return math.pi * radius * radius

# Collect available tools into a list.
tools = [circle_area]

# Bind tools to the model so it can decide to call them
# Describe the tools to the model. Binding does not execute these functions.
llm_with_tools = llm.bind_tools(tools)

# Create the person's question, then ask the model to respond or request a tool.
msg = HumanMessage(content="I have a circular garden of radius 3. What's the area?")
ai_msg = llm_with_tools.invoke([msg])

# Inspect the reply and any requested tool calls.
# getattr reads an attribute, returning None if that attribute is missing.
print("Model output Type", type(ai_msg))
print("Model output", ai_msg)
print("Tool calls (if any):", getattr(ai_msg, "tool_calls", None))

# Create the person's question, then ask the model to respond or request a tool.
msg = HumanMessage(content="I have a square garden of size 4X4. What's the area?")
ai_msg = llm_with_tools.invoke([msg])

# Inspect the reply and any requested tool calls.
# getattr reads an attribute, returning None if that attribute is missing.
print("Model output Type", type(ai_msg))
print("Model output", ai_msg)
print("Tool calls (if any):", getattr(ai_msg, "tool_calls", None))



Model output Type <class 'langchain_core.messages.ai.AIMessage'>
Model output content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 62, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dc13727eb9', 'id': 'chatcmpl-EZKa7RAdarpbsXgbjFX0NHrjNHwTK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a140d8-a740-7552-9e23-e8f3fd5c20a7-0' tool_calls=[{'name': 'circle_area', 'args': {'radius': 3}, 'id': 'call_UAeyx0G4rcApTbu038lPDcu7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 62, 'output_tokens'

In [13]:
# Handle a tool request manually. This Python code performs the calculation
# and formats the answer; it does not send the result back to the model.

# If a tool call is present, execute it and return a final answer:

# Start with no final answer selected yet. None means no value.
final = None
# Check whether the model requested any tools before trying to run them.
if getattr(ai_msg, "tool_calls", None):
    # Process each requested tool call one at a time.
    for tool_call in ai_msg.tool_calls:
        # Only run the circle_area tool that this example knows how to handle.
        if tool_call["name"] == "circle_area":
            # Read the radius from the tool arguments and convert it to a number.
            r = float(tool_call["args"]["radius"])
            # Execute the actual Python calculation using that radius.
            result = circle_area.invoke({"radius": r})
            # Insert the result into a sentence. :.2f shows two digits after the decimal point.
            final = f"The area is approximately {result:.2f} square units."
# If there were no tool requests, use the model's text answer directly.
else:
    final = ai_msg.content

# Display the selected answer. With multiple circle calls, this shows the last one.
print("Final:", final)

Final: The area of a square can be calculated using the formula:

\[
\text{Area} = \text{side} \times \text{side}
\]

For a square garden with a side length of 4 units:

\[
\text{Area} = 4 \times 4 = 16 \text{ square units}
\]

So, the area of your square garden is 16 square units.


## 7) Create Agent

In Section 6, *you* were the loop — the AI flagged a tool call, you ran it, and handed the result back. That works for one call. But what if the question needs two tools? You'd have to build that dispatch logic yourself.

`create_agent` handles that loop for you. Every agent is two things:

**Agent = Model + Harness**
* **Model** — the brain. Reads context and decides: *call a tool, or give the final answer.*
* **Harness** — the loop. Its job is to *get the model the right context at the right time* — run the requested tool, feed the result back, and repeat until the model is done.

![Agent loop](./agent_loop.png)

### Code Context
In the code below:
* `create_agent`: Builds the agent — takes a `model`, a list of `tools`, and an optional `system_prompt`.
* `tools=[circle_area, rectangle_area]`: We give the agent **two** tools. It decides which ones to call and when.
* `agent.invoke({"messages": [...]})`: The input is a dict with a `"messages"` key — not a raw list.
* `result["messages"][-1].content`: The agent returns all messages (question → tool calls → tool results → final answer). The last one is the answer.

📚 **[Docs: Agents](https://docs.langchain.com/oss/python/langchain/agents)**

In [14]:
# Create an agent: a workflow that can ask the model which tools to use,
# execute those tools, and pass their results back for an answer.
# These tool definitions replace the earlier circle_area definition.

# Import the helper that coordinates the model and available tools.
from langchain.agents import create_agent
from langchain.tools import tool
import math

# Expose the following calculation to the agent. The description explains its inputs.
@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius.

    Args:
        radius: The radius of the circle
    """
    # Circle area = pi times radius times radius.
    return math.pi * radius * radius

@tool
def rectangle_area(length: float, width: float) -> float:
    """Return the area of a rectangle given its length and width.

    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    """
    # Rectangle area = length times width.
    return length * width

# Build the agent with the existing model, both calculation tools,
# and instructions for how it should answer. This does not ask a question yet.
agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations."
)

In [15]:
# Ask one question that can require both geometry tools.

#question = "I have a circular pond with radius 3 , triangular backyard with base 5 and height 3, and a rectangular patio that is 4 by 5. What is the combined area of all three?"
question = "I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. What is the combined area ?"

# Start the agent workflow. role identifies who is speaking; content is their text.
# The result includes the conversation and any tool requests and results.
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# [-1] selects the last message, where the agent's final answer is normally stored.
print("Final Answer:", result["messages"][-1].content)

# Peek under the hood — the agent called both tools automatically
print("\n--- Agent's steps ---")
# Walk through the returned conversation to see how the answer was produced.
for msg in result["messages"]:
    # Read the Python class name to identify the kind of message.
    label = type(msg).__name__
    # Check that this message has tool calls before reading them.
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        # Show the requested tool names. The list expression collects one name per call.
        print(f"{label}: called → {[tc['name'] for tc in msg.tool_calls]}")
    # For other messages that contain text, show a short preview.
    elif msg.content:
        # Convert content to text and display only its first 80 characters.
        print(f"{label}: {str(msg.content)[:800]}")

Final Answer: To find the combined area of the circular pond and the rectangular patio, we first calculate their individual areas:

1. **Area of the circular pond**:
   \[
   \text{Area} = \pi \times r^2 = \pi \times 3^2 \approx 28.27 \text{ square units}
   \]

2. **Area of the rectangular patio**:
   \[
   \text{Area} = \text{length} \times \text{width} = 4 \times 5 = 20 \text{ square units}
   \]

Now, we add the two areas together:
\[
\text{Combined Area} = 28.27 + 20 = 48.27 \text{ square units}
\]

Thus, the combined area of the circular pond and the rectangular patio is approximately **48.27 square units**.

--- Agent's steps ---
HumanMessage: I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. What is the combined area ?
AIMessage: called → ['circle_area', 'rectangle_area']
ToolMessage: 28.274333882308138
ToolMessage: 20.0
AIMessage: To find the combined area of the circular pond and the rectangular patio, we first calculate their individual areas:

1. 

![Diagram](./images/07_img.png)

In [16]:
# Build an interactive chatbot that remembers earlier turns within a thread.
# Run the previous model and tool definition cells first.

## Simple Q/A Chatbot
# Import temporary conversation storage held in this Python process's memory.
# This memory is lost when the process or notebook kernel restarts.
from langgraph.checkpoint.memory import InMemorySaver

# Create an agent with the geometry tools and conversation storage.
agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations.",
    # Save conversation state between calls to this agent while it remains in memory.
    checkpointer=InMemorySaver()
)

# Use the same thread ID for follow-up questions in the same conversation.
# A different thread ID starts a separate conversation history.
config = {"configurable": {"thread_id": "geometry-chat"}}

print("Geometry Chatbot ready! Type 'quit' to exit.\n")

# Keep accepting questions until an exit command breaks the loop.
# This cell stays running while it waits for keyboard input.
while True:
    # Ask for a question and remove extra spaces from its beginning and end.
    user_input = input("You: ").strip()
    # Skip empty input and return to the beginning of the loop.
    if not user_input:
        continue
    # Accept quit or exit in any capitalization, then stop the loop.
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break

    # Send the new question using the same thread so previous turns are available.
    # The agent can use the geometry tools when needed.
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    # Display the most recent answer, then wait for another question.
    print(f"Agent: {result['messages'][-1].content}\n")

Geometry Chatbot ready! Type 'quit' to exit.

Agent: The area of a circle with a radius of 4 is approximately \( 50.27 \) square units.

Agent: Today's date is October 4, 2023.

Agent: The area of a circle with a new radius of 6 (after adding 2 to the previous radius of 4) is approximately \( 113.10 \) square units.

Goodbye!
